In [ ]:
# Cell 1: setup
import sys
from pathlib import Path

import pandas as pd
import nflreadpy as nfl

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.fetch import fetch_schedules

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

In [ ]:
# Cell 2: load both datasets
schedules = fetch_schedules()
team_stats = nfl.load_team_stats(seasons=[2025, 2026]).to_pandas()
schedules["played"] = schedules["home_score"].notna() & schedules["away_score"].notna()
print(schedules.shape, team_stats.shape)

In [ ]:
# Cell 3: games per season and type
schedules.groupby(["season", "game_type"]).agg(
    games=("game_id", "count"), played=("played", "sum")
)

In [ ]:
# Cell 4: home field, margin and total by season
played = schedules[schedules["played"]].copy()
played["home_win"] = played["home_score"] > played["away_score"]
played["margin"] = played["home_score"] - played["away_score"]
played["total"] = played["home_score"] + played["away_score"]

played.groupby("season").agg(
    games=("game_id", "count"),
    home_win_rate=("home_win", "mean"),
    avg_margin=("margin", "mean"),
    avg_total=("total", "mean"),
)

In [ ]:
# Cell 5: points for and against by team, best point differential first
home = played[["season", "home_team", "home_score", "away_score"]].rename(
    columns={"home_team": "team", "home_score": "pf", "away_score": "pa"})
away = played[["season", "away_team", "away_score", "home_score"]].rename(
    columns={"away_team": "team", "away_score": "pf", "home_score": "pa"})

team_games = pd.concat([home, away])
(team_games.groupby(["season", "team"])
    .agg(games=("pf", "size"), pf=("pf", "mean"), pa=("pa", "mean"))
    .assign(diff=lambda d: d["pf"] - d["pa"])
    .sort_values("diff", ascending=False)
    .head(10))

In [ ]:
# Cell 6: team offense and defense efficiency in 2026
ts26 = team_stats[team_stats["season"] == 2026]
(ts26.groupby("team")[["passing_epa", "rushing_epa", "passing_yards", "rushing_yards"]]
    .mean()
    .sort_values("passing_epa", ascending=False)
    .head(10))

In [ ]:
# Cell 7: confirm the join and preview which stats track scoring
merged = team_stats.merge(
    schedules[["game_id", "home_team", "away_team", "home_score", "away_score"]],
    on="game_id", how="left", validate="m:1",
)
print("Unmatched game_ids:", merged["home_team"].isna().sum())

merged["is_home"] = merged["team"] == merged["home_team"]
merged["points_for"] = merged["home_score"].where(merged["is_home"], merged["away_score"])

merged[["points_for", "passing_epa", "rushing_epa", "passing_yards", "rushing_yards"]].corr()["points_for"]

In [ ]:
# Cell 8: plot (needs matplotlib)
# Histogram of total points per game (home score + away score).
# Uses the `played` DataFrame from Cell 4, so run Cells 1 to 4 first.
# pandas calls matplotlib under the hood, which is why it must be installed.
played["total"].plot.hist(
    bins=20,                      # 20 equal-width buckets across the range of totals
    title="Total points per game",
)

In [ ]:
# Show the 10 most recent games that have final scores.
# `schedules` comes from Cell 2, where the boolean column `played` marks games with both scores.
recent = schedules[schedules["played"]].copy()

# Convert gameday to real dates so sorting is chronological, not alphabetical.
recent["gameday"] = pd.to_datetime(recent["gameday"])

(recent.sort_values(["gameday", "gametime"])   # order by date, then kickoff time
       .tail(10)                               # keep the last 10 rows, the newest games
       [["game_id", "week", "gameday", "gametime",
         "away_team", "away_score", "home_team", "home_score"]])

In [ ]:
# Compare scored games in the schedule against game_ids present in team_stats.
# Team stats can publish later than final scores, so it may lag behind.
stats_game_ids = set(team_stats["game_id"])
scored = schedules[schedules["played"]]

# Scored games that have no row in team_stats yet.
missing = scored[~scored["game_id"].isin(stats_game_ids)]
print("Played games missing from team_stats:", len(missing))
missing[["game_id", "week", "gameday"]]

In [ ]:
# Pick the latest season in the schedule data (2026) and keep only games with final scores.
CURRENT_SEASON = schedules["season"].max()
cur = schedules[(schedules["season"] == CURRENT_SEASON) & schedules["played"]]

# Each game involves two teams, so build one view from the home team's side...
home_view = pd.DataFrame({
    "team": cur["home_team"],
    "opponent": cur["away_team"],
    "location": "home",
    "week": cur["week"],
    "gameday": cur["gameday"],
    "points_scored": cur["home_score"].astype(int),    # home team's points
    "points_allowed": cur["away_score"].astype(int),   # points the opponent scored
})

# ...and one from the away team's side (the same columns with the roles swapped).
away_view = pd.DataFrame({
    "team": cur["away_team"],
    "opponent": cur["home_team"],
    "location": "away",
    "week": cur["week"],
    "gameday": cur["gameday"],
    "points_scored": cur["away_score"].astype(int),
    "points_allowed": cur["home_score"].astype(int),
})

# Stack the two views: every game now appears twice, once per team.
team_games = pd.concat([home_view, away_view], ignore_index=True)

# Add a W/L/T label from the score difference.
diff = team_games["points_scored"] - team_games["points_allowed"]
team_games["result"] = diff.map(lambda d: "W" if d > 0 else ("L" if d < 0 else "T"))

# Show all rows instead of pandas' default truncation, sorted by team then week.
pd.set_option("display.max_rows", 200)
team_games.sort_values(["team", "week"]).reset_index(drop=True)

In [ ]:
# Collapse team_games to one row per team.
summary = (
    team_games.sort_values(["team", "week"])           # sort first so opponents list in week order
    .groupby("team")
    .agg(
        games=("week", "size"),                        # games played so far
        points_scored=("points_scored", "sum"),        # total points the team scored
        points_allowed=("points_allowed", "sum"),      # total points the team gave up
        opponents=("opponent", lambda s: ", ".join(s)),  # opponents in the order faced
    )
    .assign(point_diff=lambda d: d["points_scored"] - d["points_allowed"])  # scored minus allowed
    .sort_values("point_diff", ascending=False)        # best point differential first
)
summary

In [ ]:
# Plotly weekly points chart with hover tooltips, home/away markers, and a reading guide table below.
# Reads `team_games` from Cell 11 (current season, one row per team per game).
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def plot_team_hover(team="ALL"):
    # Filter to one team, or keep every row for "ALL".
    data = team_games if team == "ALL" else team_games[team_games["team"] == team]

    # Two stacked panels: the chart on top (row 1) and the reference table below (row 2).
    fig = make_subplots(
        rows=2, cols=1,
        specs=[[{"type": "xy"}], [{"type": "table"}]],
        row_heights=[0.62, 0.38],
        vertical_spacing=0.08,
    )

    if team == "ALL":
        # League view: average points per team in each week, plus how many teams played.
        by_week = data.groupby("week").agg(
            scored=("points_scored", "mean"),
            allowed=("points_allowed", "mean"),
            n=("team", "size"),
        ).reset_index()
        for col, color, name in [("scored", "royalblue", "scored"), ("allowed", "crimson", "allowed")]:
            fig.add_trace(go.Scatter(
                x=by_week["week"], y=by_week[col], mode="lines+markers",
                name=f"Avg points {name}", line=dict(color=color),
                customdata=by_week[["n"]],
                hovertemplate="Week %{x}<br>Avg points " + name
                              + ": %{y:.1f}<br>Teams: %{customdata[0]}<extra></extra>",
            ), row=1, col=1)
    else:
        # Team view: one point per game, with opponent, location, score and date on hover.
        d = data.sort_values("week")

        # One marker symbol per game: circle for home, diamond for away.
        symbols = d["location"].map({"home": "circle", "away": "diamond"}).tolist()

        for col, color, name in [("points_scored", "royalblue", "scored"),
                                 ("points_allowed", "crimson", "allowed")]:
            fig.add_trace(go.Scatter(
                x=d["week"], y=d[col], mode="lines+markers",
                name=f"Points {name}", line=dict(color=color),
                marker=dict(symbol=symbols, size=11),
                # customdata columns: 0 opponent, 1 location, 2 result, 3 date, 4 points scored, 5 points allowed
                customdata=d[["opponent", "location", "result", "gameday",
                              "points_scored", "points_allowed"]],
                hovertemplate=("Week %{x} (%{customdata[3]})"
                               "<br>%{customdata[1]} vs %{customdata[0]}"
                               "<br>Score: %{customdata[4]} to %{customdata[5]} (%{customdata[2]})"
                               "<br>Points " + name + ": %{y}<extra></extra>"),
            ), row=1, col=1)

        # Legend-only entries that explain the symbols (no data, so nothing is drawn on the plot).
        for sym, label in [("circle", "Home game"), ("diamond", "Away game")]:
            fig.add_trace(go.Scatter(
                x=[None], y=[None], mode="markers",
                marker=dict(symbol=sym, size=11, color="gray"),
                name=label, hoverinfo="skip",
            ), row=1, col=1)

    # Mean (dashed) and median (dotted) as flat lines across the weeks shown.
    x_min, x_max = data["week"].min(), data["week"].max()
    for col, color, name in [("points_scored", "royalblue", "scored"),
                             ("points_allowed", "crimson", "allowed")]:
        for stat, dash, val in [("mean", "dash", data[col].mean()),
                                ("median", "dot", data[col].median())]:
            fig.add_trace(go.Scatter(
                x=[x_min, x_max], y=[val, val], mode="lines",
                line=dict(color=color, dash=dash, width=1.5),
                name=f"{name} {stat}: {val:.1f}",
                hovertemplate=f"{name} {stat}: {val:.1f}<extra></extra>",
            ), row=1, col=1)

    # Reading guide: how to interpret a gap between median and mean.
    # Plotly table cells do not wrap text on their own, so <br> marks the line breaks.
    fig.add_trace(go.Table(
        header=dict(values=["Series", "Median above mean (left-skewed)", "Mean above median (right-skewed)"],
                    align="left", fill_color="lightgray"),
        cells=dict(
            values=[
                ["Points scored", "Points allowed"],
                ["Mostly solid scoring, with one or two very weak offensive games",
                 "Usually gives up a lot, with one or two dominant defensive games (a shutout, say)"],
                ["Mostly modest scoring, with one or two explosive games",
                 "Usually holds opponents down, with one or two blowout losses"],
            ],
            align="left", height=32,
        ),
    ), row=2, col=1)

    # Axis settings apply to the chart panel only.
    fig.update_xaxes(title_text="Week", dtick=1, row=1, col=1)                  # one tick per week
    fig.update_yaxes(title_text="Points", rangemode="tozero", row=1, col=1)     # points axis starts at zero
    fig.update_layout(
        title=f"{team}: points by week (season {CURRENT_SEASON})",
        hovermode="closest",
        height=800,
    )
    return fig


# Draw one team to confirm the layout. Change the code to any team in your data.
plot_team_hover("KC").show()

In [ ]:
# Dropdown control. interact calls show_team again whenever the selection changes.
import ipywidgets as widgets


def show_team(team):
    # Build the figure for the chosen team and display it.
    plot_team_hover(team).show()


widgets.interact(
    show_team,
    team=["ALL"] + sorted(team_games["team"].unique()),   # "ALL" plus each team code
)

In [ ]:
# Read straight from the SQLite database instead of downloading from nflverse.
# ROOT was defined in Cell 1 as the repo root folder.
import sqlite3

conn = sqlite3.connect(ROOT / "data" / "nfl.db")

# The last 10 games with final scores, newest at the bottom.
pd.read_sql_query(
    """
    SELECT game_id, week, gameday, away_team, away_score, home_team, home_score
    FROM games
    WHERE home_score IS NOT NULL
    ORDER BY gameday, gametime
    LIMIT 10 OFFSET (SELECT COUNT(*) FROM games WHERE home_score IS NOT NULL) - 10
    """,
    conn,
)